# Module B: Optimization Implementation Report
## IITGN Connect — CS432 Databases, Assignment 2

**Project:** IITGN Connect — A College Social Media Platform  
**Course:** CS 432 — Databases (Semester II, 2025-2026)  
**Institute:** Indian Institute of Technology, Gandhinagar  

---

## Table of Contents
1. Schema Design
2. Local Environment Setup & Data Management
3. API & UI Development (Member Portfolio)
4. Security — Session Validation & RBAC
5. Indexing Strategy
6. Performance Benchmarking
7. EXPLAIN Plan Analysis
8. Video Demonstration Link

---
## 1. Schema Design

### Overview
IITGN Connect uses a MySQL relational database with **21 tables** organized into core system tables and project-specific feature tables. The design follows strict normalization principles to avoid data duplication.

### Core System Tables
| Table | Purpose | Key Constraints |
|-------|---------|-----------------|
| **Member** | Central user table (all member types) | PK: `MemberID`, UNIQUE: `Username`, `Email` |
| **Student** | Student-specific attributes | PK/FK: `MemberID` → Member, ON DELETE CASCADE |
| **Professor** | Professor-specific attributes | PK/FK: `MemberID` → Member, ON DELETE CASCADE |
| **Alumni** | Alumni-specific attributes | PK/FK: `MemberID` → Member, ON DELETE CASCADE |
| **Organization** | Organization-specific attributes | PK/FK: `MemberID` → Member, ON DELETE CASCADE |

### Project-Specific Tables
| Table | Purpose | Key Relationships |
|-------|---------|-------------------|
| **CampusGroup** | Groups/clubs/communities | FK: `AdminID` → Member |
| **GroupMembership** | Many-to-many: Members ↔ Groups | Composite PK: `(GroupID, MemberID)` |
| **Post** | User-generated content | FK: `AuthorID` → Member, `GroupID` → CampusGroup |
| **Comment** | Comments on posts | FK: `PostID` → Post, `AuthorID` → Member |
| **PostLike** | Like/unlike on posts | Composite PK: `(PostID, MemberID)` |
| **Poll / PollOption / PollVote** | Polls with options and votes | Cascading deletes through Poll → Options → Votes |
| **JobPost** | Alumni job postings | FK: `AlumniID` → Alumni |
| **ReferralRequest** | Student referral requests | FK: `StudentID` → Student, `TargetAlumniID` → Alumni |
| **Course / Enrollment** | Course management | Composite PK on Enrollment: `(StudentID, CourseID, Semester)` |
| **ClassAttendance / MessAttendance** | Attendance tracking | FK to Student and Course |
| **ProfileClaimQuestion / ProfileClaimVote** | Q&A claims with voting | FK cascades for cleanup |
| **AuditLog** | Security audit trail | Standalone logging table |

### Data Integrity Strategy
- **ISA Hierarchy**: Member is the supertype; Student/Professor/Alumni/Organization are subtypes linked via shared PK with `ON DELETE CASCADE`. Deleting a Member automatically removes the subtype row and all dependent data.
- **No Duplication**: Login credentials are stored only in the `Member` table. Subtype tables reference `MemberID` as both PK and FK.
- **Cascade Deletes**: All FK relationships use `ON DELETE CASCADE` to maintain referential integrity when members or groups are deleted.

### Entity-Relationship Summary
```
Member (1) ──ISA──> Student / Professor / Alumni / Organization
Member (1) ───< (M) GroupMembership (M) >─── (1) CampusGroup
Member (1) ───< (M) Post ───< (M) Comment
                     Post ───< (M) PostLike
Member (1) ───< (M) Poll ───< (M) PollOption ───< (M) PollVote
Alumni (1) ───< (M) JobPost
Student (1) ───< (M) ReferralRequest >─── (1) Alumni
Student (1) ───< (M) ClassAttendance >─── (1) Course
Student (1) ───< (M) MessAttendance
```

---
## 2. Local Environment Setup & Data Management

### Environment Setup
- **Database initialization:** `app/backend/seed.py` creates all core and project-specific tables in the `iitgn_connect` database.
- **Schema scripts:** `sql/schema.sql` documents the full DDL used by the local system.
- **Indexes:** `sql/indexes.sql` contains the index strategy applied for optimization.

### Core vs Project Tables
- Core data (members, credentials, roles, group membership) is centralized in `Member` and related subtype tables.
- Project-specific data (posts, polls, jobs, attendance, referrals) references core tables via FKs without duplicating credentials.

### Integrity on Create/Delete
- Member creation inserts into `Member` and its subtype table (Student/Alumni/Professor/Organization).
- Deletions cascade through FK constraints (`ON DELETE CASCADE`) to keep dependent tables consistent.

---
## 3. API & UI Development (Member Portfolio)

### Backend APIs
- Implemented in Flask (`app/backend/app.py`) with route modules under `app/backend/routes/`.
- CRUD endpoints cover posts, groups, polls, jobs, referrals, attendance, members, and settings.

### Frontend UI
- Built with Vite + React in `app/iitgn-connect`.
- Auth flow and API calls are centralized in `src/contexts/AuthContext.jsx` and `src/api.js`.
- Member Portfolio is implemented in the Profile page and requires an authenticated session.

### Session Validation
- All APIs (except login/register) require a valid JWT; unauthenticated requests return 401.

---
## 4. Security — Session Validation & RBAC

### Authentication Mechanism
IITGN Connect uses **JWT (JSON Web Tokens)** for stateless session management:

1. **Login Flow**: `POST /api/auth/login` validates credentials against bcrypt-hashed passwords in the `Member` table, then issues a JWT token with 24-hour expiry.
2. **Session Validation**: Every API endpoint (except login/register) requires a valid JWT via the `@jwt_required()` decorator. The `GET /api/auth/isAuth` endpoint allows the frontend to verify token validity and retrieve the user's role and expiry time.
3. **Token Structure**: The JWT `identity` is the `MemberID`, ensuring all subsequent queries are scoped to the authenticated user.

### Role-Based Access Control (RBAC)

| Role | Permissions |
|------|------------|
| **Admin** | Full CRUD on all members, groups, posts. Can view stats dashboard. Can delete any member or group. |
| **Regular User** | Can create/edit/delete their **own** posts, comments, and profile. Can view other profiles (read-only). Can join/leave groups. |
| **Alumni** | Regular permissions + can create job postings and manage referral requests. |
| **Student** | Regular permissions + can submit referral requests and record attendance. |

### RBAC Implementation
- **`admin_required()` decorator**: Applied to all `/api/admin/*` endpoints. Checks the `IsAdmin` boolean on the `Member` table before allowing the request.
- **Ownership checks**: Edit/delete operations on posts, comments, and profile claims verify that `AuthorID == current_user_id`.
- **Type-based restrictions**: Job posting is restricted to Alumni (`MemberType = 'Alumni'`). Referral requests are restricted to Students.

### Security Audit Logging
All data-modifying API calls (POST, PUT, DELETE) are logged to:
1. **`logs/audit.log`** — a file-based log with timestamps, usernames, actions, endpoints, IP addresses, and operation details.
2. **`AuditLog` database table** — stores the same information for queryable audit trail.

**Log format:**
```
[2026-03-19 10:30:45] [INFO] [USER:laksh_jain] [ACTION:POST] [ENDPOINT:/api/posts/] [IP:127.0.0.1] — Created new post in group 4
```

### Detecting Unauthorized Direct DB Modifications
- All legitimate API operations are logged with `IsAuthorized = TRUE` in the AuditLog table.
- Any data changes found in the database that do **not** have a corresponding AuditLog entry can be flagged as unauthorized modifications.
- The audit log can be cross-referenced with actual table row counts and timestamps to detect discrepancies.

---
## 5. Indexing Strategy

### Approach
We analyzed every SQL query in the Flask API route handlers and identified columns used in:
- **WHERE** clauses (point lookups and range filters)
- **JOIN** conditions (FK → PK lookups)
- **ORDER BY** clauses (sorting)
- **Correlated subqueries** (e.g., `COUNT(*)` in post listings)

### Indexes Created

| # | Index Name | Table | Column(s) | Reason |
|---|-----------|-------|-----------|--------|
| 1 | `idx_post_authorid` | Post | AuthorID | Profile page: posts by author, JOIN to Member |
| 2 | `idx_post_groupid` | Post | GroupID | Group feed: WHERE GroupID = ? |
| 3 | `idx_post_groupid_createdat` | Post | (GroupID, CreatedAt DESC) | Group posts sorted by date — avoids filesort |
| 4 | `idx_post_createdat` | Post | CreatedAt DESC | Global feed: ORDER BY CreatedAt DESC |
| 5 | `idx_post_authorid_createdat` | Post | (AuthorID, CreatedAt DESC) | Profile: recent posts by author |
| 6 | `idx_comment_postid_createdat` | Comment | (PostID, CreatedAt ASC) | Comments per post, sorted by time |
| 7 | `idx_comment_postid` | Comment | PostID | Comment count subquery, CASCADE deletes |
| 8 | `idx_comment_authorid` | Comment | AuthorID | JOIN to Member for comment author name |
| 9 | `idx_postlike_memberid` | PostLike | MemberID | "Liked posts by user" reverse lookup |
| 10 | `idx_groupmembership_memberid` | GroupMembership | MemberID | User's groups lookup (profile, feed) |
| 11 | `idx_campusgroup_adminid` | CampusGroup | AdminID | JOIN to Member for group admin name |
| 12 | `idx_member_name` | Member | Name | Member search: ORDER BY Name |
| 13 | `idx_member_membertype` | Member | MemberType | Member filter: WHERE MemberType = ? |
| 14 | `idx_poll_createdat` | Poll | CreatedAt DESC | Poll listing sorted by date |
| 15 | `idx_poll_creatorid` | Poll | CreatorID | JOIN to Member for poll creator |
| 16 | `idx_polloption_pollid` | PollOption | PollID | Options per poll |
| 17 | `idx_pollvote_memberid` | PollVote | MemberID | User's vote check |
| 18 | `idx_jobpost_postedat` | JobPost | PostedAt DESC | Job listing sorted by date |
| 19 | `idx_jobpost_alumniid` | JobPost | AlumniID | JOIN to Alumni for poster info |
| 20 | `idx_referral_targetalumniid_requestedat` | ReferralRequest | (TargetAlumniID, RequestedAt DESC) | Alumni referral dashboard |
| 21 | `idx_referral_studentid_requestedat` | ReferralRequest | (StudentID, RequestedAt DESC) | Student referral history |
| 22 | `idx_classattendance_studentid_date` | ClassAttendance | (StudentID, RecordDate) | Attendance by student + date range |
| 23 | `idx_classattendance_courseid` | ClassAttendance | CourseID | JOIN to Course |
| 24 | `idx_messattendance_studentid_date` | MessAttendance | (StudentID, RecordDate) | Mess attendance by student + date |
| 25 | `idx_profileclaim_memberid` | ProfileClaimQuestion | MemberID | Claims per member profile |
| 26 | `idx_enrollment_courseid` | Enrollment | CourseID | Course detail lookups |

### Index Types Used
- **Single-column indexes**: For simple WHERE/JOIN lookups (e.g., `idx_post_authorid`)
- **Composite indexes**: For queries that filter AND sort on different columns (e.g., `idx_post_groupid_createdat` covers `WHERE GroupID = ? ORDER BY CreatedAt DESC`)
- **Descending indexes**: MySQL 8.0+ supports DESC index key parts, used for `ORDER BY ... DESC` optimization

### Existing Indexes (from schema constraints)
- All **PRIMARY KEY** columns have implicit B-tree indexes
- **UNIQUE** constraints on `Member.Username` and `Member.Email` create implicit indexes
- Composite PKs like `(PostID, MemberID)` on PostLike serve as covering indexes for count queries

---
## 6. Performance Benchmarking

The benchmark script (`benchmark.py`) measures query execution times **before** and **after** applying indexes. Each query is run 100 times and averaged for statistical reliability.

### Benchmark Queries
The following representative queries were benchmarked — they cover the most performance-critical API endpoints:

In [5]:
# Run the benchmark and display results
import subprocess, json, os, sys
from pathlib import Path

BASE_DIR = Path("app/backend").resolve()

# Run the benchmark script
result = subprocess.run(
    [sys.executable, "benchmark.py"],
    capture_output=True,
    text=True,
    cwd=str(BASE_DIR),
    timeout=120,
 )
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr[-500:])


STDERR: enticator.authenticate(
  File "C:\Users\Parth\AppData\Roaming\Python\Python310\site-packages\mysql\connector\authentication.py", line 381, in authenticate
    ok_pkt = self._handle_server_response(sock, pkt)
  File "C:\Users\Parth\AppData\Roaming\Python\Python310\site-packages\mysql\connector\authentication.py", line 287, in _handle_server_response
    raise get_exception(pkt)
mysql.connector.errors.ProgrammingError: 1045 (28000): Access denied for user 'root'@'localhost' (using password: YES)



In [6]:
# Load and display benchmark results as a table
import json
import pandas as pd
from pathlib import Path

results_path = Path("app/backend/benchmark_results.json")

try:
    with open(results_path, "r", encoding="utf-8") as f:
        results = json.load(f)

    rows = []
    for r in results:
        rows.append({
            "Query": r["name"],
            "Before (ms)": round(r["before_avg_ms"], 4),
            "After (ms)": round(r["after_avg_ms"], 4),
            "Speedup": f"{r['speedup_pct']:.1f}%",
        })
    
    df = pd.DataFrame(rows)
    display(df.style.set_caption("Query Performance: Before vs After Indexing"))
except FileNotFoundError:
    print("Run benchmark.py first to generate results.")

KeyError: 'before_avg_ms'

In [ ]:
# Display benchmark charts
from IPython.display import Image, display as ipy_display
import glob
import os

chart_files = sorted(glob.glob("app/backend/benchmarks/*.png"))
for chart in chart_files:
    print(f"\n--- {os.path.basename(chart)} ---")
    ipy_display(Image(filename=chart, width=700))

---
## 7. EXPLAIN Plan Analysis

Below we show the EXPLAIN output for key queries **before** and **after** indexing to demonstrate how the MySQL query planner's access strategy changes.

In [ ]:
# Display EXPLAIN plans before vs after for each query
import json
from pathlib import Path

results_path = Path("app/backend/benchmark_results.json")
try:
    with open(results_path, "r", encoding="utf-8") as f:
        results = json.load(f)

    for r in results:
        print("=" * 80)
        print(f"Query: {r['name']}")
        print(f"SQL: {r['sql'][:120]}...")
        print()
        
        print("EXPLAIN BEFORE indexing:")
        if r.get("explain_before"):
            for row in r["explain_before"]:
                access = row.get("type", row.get("access_type", "N/A"))
                key = row.get("key", "NULL")
                rows_est = row.get("rows", "N/A")
                extra = row.get("Extra", "")
                table = row.get("table", "N/A")
                print(f"  table={table}  type={access}  key={key}  rows={rows_est}  Extra={extra}")
        
        print("\nEXPLAIN AFTER indexing:")
        if r.get("explain_after"):
            for row in r["explain_after"]:
                access = row.get("type", row.get("access_type", "N/A"))
                key = row.get("key", "NULL")
                rows_est = row.get("rows", "N/A")
                extra = row.get("Extra", "")
                table = row.get("table", "N/A")
                print(f"  table={table}  type={access}  key={key}  rows={rows_est}  Extra={extra}")
        
        print(f"\nSpeedup: {r['speedup_pct']:.1f}%")
        print()
except FileNotFoundError:
    print("Run benchmark.py first to generate results.")

### EXPLAIN Plan Interpretation

| Access Type | Meaning | Performance |
|------------|---------|-------------|
| `ALL` | Full table scan — reads every row | Slowest |
| `index` | Full index scan — reads entire index | Better than ALL |
| `range` | Index range scan — reads a subset of index | Good |
| `ref` | Non-unique index lookup | Good |
| `eq_ref` | Unique index lookup (one row) | Excellent |
| `const` | Primary key / unique lookup (one row) | Best |

**Key improvements after indexing:**
- Queries that previously used `type=ALL` (full table scan) shift to `type=ref` or `type=range` (index-based access)
- The `rows` estimate decreases significantly, meaning fewer rows are examined
- `Using filesort` disappears from `Extra` when composite indexes cover the ORDER BY clause
- `Using index` appears in `Extra` for covering indexes where all needed columns are in the index

---
## 8. Video Demonstration

> **Video Link:** 
> 
 > 
 > The video covers:
 > 1. UI & API functionality walkthrough
 > 2. RBAC enforcement (Admin vs Regular User login)
 > 3. Security logging mechanism demonstration